# conv-output-shape composite — cx11: conv2d output shape under stride downsample

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-output-shape`, `conv-stride-downsample`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-output-shape"
DD_ATOM_IDS = ["conv-output-shape", "conv-stride-downsample"]
DD_SUBTOPICS = ["CNN: Conv output shape", "CNN: Stride downsample arithmetic"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The 2-D conv output-shape formula expressed via the stride-downsample arithmetic:
```
H_out = (H + 2*PH - KH) // SH + 1
W_out = (W + 2*PW - KW) // SW + 1
```

When `stride > 1`, the floor-division `// SH` is doing the actual downsampling. The `+ 1` captures the leading window (this is the `conv-stride-downsample` half of the formula — without `+1` you'd be off by one for *every* stride).

**Two flavours of stride downsample.**
- *Naive intuition*: 'stride 2 halves the spatial size'. WRONG without padding. For `H=32, K=3, S=2, P=0` you get `(32-3)//2 + 1 = 15`, NOT 16.
- *Same-pad downsample* (`K=3, P=1, S=2`): `(32 + 2 - 3)//2 + 1 = 16`. The padding term cancels the kernel shrink, leaving exactly `H // S`. This is the ResNet pattern.

This drill exercises both: predict the output shape AND a helper that, given input length and stride, returns the analytic 'clean halving' padding (i.e. the `P` such that `H_out == H / S` for a stride-`S`, kernel-3 conv).

### Composite Exercise — conv2d output shape under stride downsample

**Atoms exercised together**: `conv-output-shape`, `conv-stride-downsample`

Implement two functions.

1. `cx11_outshape_with_stride(input_shape, out_channels, kernel_size, stride, padding)`:
   - Same signature as cx10 (and same formula). Return `(B, OC, H_out, W_out)`.

2. `cx11_clean_halve_padding(K, S)`:
   - Given odd kernel `K` and stride `S`, return the integer `P` such that for even input lengths `H`, `cx11_outshape_with_stride` gives `H_out == H // S`.
   - Closed form: `P = (K - 1) // 2`. The reasoning: `(H + 2P - K) // S + 1` reduces to `H // S` when `2P = K - 1`.
   - You may assume `K` is odd.

The test cross-checks the forward function against `nn.Conv2d` and verifies the helper produces clean halving (or thirding, etc.) for several `(K, S)` combos.

In [ ]:
def cx11_outshape_with_stride(input_shape, out_channels, kernel_size, stride, padding):
    B, IC, H, W = input_shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    # Atom A (conv-output-shape) — atom B (conv-stride-downsample) lives in the // SH + 1.
    H_out = (H + 2 * PH - KH) // SH + 1
    W_out = (W + 2 * PW - KW) // SW + 1
    return (B, out_channels, H_out, W_out)

def cx11_clean_halve_padding(K, S):
    # Solve (H + 2P - K) // S + 1 == H // S for even H.
    # When 2P == K - 1: (H + K - 1 - K) // S + 1 = (H - 1) // S + 1 = H // S (for H % S == 0).
    return (K - 1) // 2


<details><summary>Show solution — cx11</summary>

```python
def cx11_outshape_with_stride(input_shape, out_channels, kernel_size, stride, padding):
    B, IC, H, W = input_shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    # Atom A (conv-output-shape) — atom B (conv-stride-downsample) lives in the // SH + 1.
    H_out = (H + 2 * PH - KH) // SH + 1
    W_out = (W + 2 * PW - KW) // SW + 1
    return (B, out_channels, H_out, W_out)

def cx11_clean_halve_padding(K, S):
    # Solve (H + 2P - K) // S + 1 == H // S for even H.
    # When 2P == K - 1: (H + K - 1 - K) // S + 1 = (H - 1) // S + 1 = H // S (for H % S == 0).
    return (K - 1) // 2
```

The `+ 1` in the output-shape formula is the entire `conv-stride-downsample` atom — without it you'd be off by one for every stride. The clean-halve helper exploits the algebra: setting `2P = K - 1` collapses the formula to `(H - 1) // S + 1`, which equals `H // S` when `H` is divisible by `S`. This is why ResNet-style downsampling uses K=3, P=1, S=2: spatial dims halve cleanly with no off-by-one.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx11'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx11',
        'subtopics': ["CNN: Conv output shape", "CNN: Stride downsample arithmetic"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()